# CrocoDashChecks

CrocoDash `main`, checked end to end on NCAR's systems.
[How it works](how_it_works.md) · [Performance](performance.ipynb)

```{toctree}
:hidden:
performance
how_it_works
integration
```

In [ ]:
import checks
from checks import FAIL, NOT_RUN, OK, WARN

testing = checks.results("testing")  # weekly: every step on the matrix's testing rows
performance = checks.results("performance")  # monthly: the performance rows and sweeps
runs = checks.load(checks.CESM_RUNS)  # weekly: CESM cases on the rows marked for them
health = checks.load(checks.HEALTH)  # weekly: every data access method
grids = checks.matrix("testing")

In [ ]:
def summary_row(name, kind, results, counts):
    """A check: how many of its parts pass, when it ran, and the CrocoDash it tested."""
    if results is None:
        return [name, f"{WARN} No results yet", "", ""]
    ok, total = counts(results)
    result = f"{OK if ok == total else FAIL} {ok}/{total} pass"
    if checks.is_stale(results["date"], kind):
        result += f" ({WARN} {checks.age_days(results['date'])} days old)"
    crocodash = checks.commit(results.get("crocodash"), "CrocoDash")
    return [name, result, checks.when(results["date"]), crocodash]


def step_counts(results):
    marks = [mark for mark, _ in results["outcomes"].values()]
    return marks.count(OK), len(marks)


checks.show(
    checks.table(
        ["Check", "Result", "Last run", "CrocoDash"],
        [
            summary_row("[Testing](#matrix)", "testing", testing, step_counts),
            summary_row("[CESM runs](#cesm)", "cesm", runs, checks.cesm_counts),
            summary_row(
                "[Data access](#data-access)", "health", health, checks.health_counts
            ),
            summary_row(
                "[Performance](performance.ipynb)",
                "performance",
                performance,
                step_counts,
            ),
        ],
    )
)

(matrix)=
## Test matrix

The testing rows of
[`matrix.yaml`](https://github.com/CROCODILE-CESM/CrocoDashChecks/blob/main/integration/matrix.yaml),
weekly: **Case** is `crocodash create`, every forcing included; **CESM**, MOM_interface's
regional tests on the row (blank is off). Performance: every step of the monthly performance
rows.

In [ ]:
from statistics import median

SUITES = {"testing": testing, "performance": performance}
outcomes = {suite: r["outcomes"] if r else {} for suite, r in SUITES.items()}

# A step has changed if its time or peak memory moved by more than a quarter, and by more
# than this, against the median of its suite's past runs.
MIN_CHANGE = {"time": 5.0, "memory": 1.0}  # seconds, GB
SLOWER, FASTER = "▲", "▼"


def measures(case):
    """A passed step's time (s) and peak memory (GB)."""
    found = {"time": case["stats"]["mean"]}
    if "peak_mb" in case["extra_info"]:
        found["memory"] = case["extra_info"]["peak_mb"] / 1024
    return found


def speed_changes(suite):
    """The suite's past runs, and {test id: [(time or memory, before, now)]} for
    its steps that changed."""
    past, changed = checks.past_results(suite, 4), {}
    for test_id, case in (SUITES[suite] or {"cases": {}})["cases"].items():
        for what, now in measures(case).items():
            before = [
                measures(run["cases"][test_id]).get(what)
                for run in past
                if test_id in run["cases"]
            ]
            before = [b for b in before if b is not None]
            if before and abs(now - median(before)) > max(
                0.25 * median(before), MIN_CHANGE[what]
            ):
                changed.setdefault(test_id, []).append((what, median(before), now))
    return past, changed


changes = {suite: speed_changes(suite) for suite in SUITES}


def moved(suite, test_ids):
    """▲ if any of these steps got slower or bigger, ▼ if they only got faster or smaller."""
    changed = changes[suite][1]
    up = [now > before for t in test_ids for _, before, now in changed.get(t, [])]
    return SLOWER if any(up) else FASTER if up else ""


def tally(marks):
    """'✅ 9/9', or ❌ with how many passed."""
    if not marks:
        return WARN
    return f"{OK if marks.count(OK) == len(marks) else FAIL} {marks.count(OK)}/{len(marks)}"


def step(test_id):
    """A testing step's mark: passed, failed, skipped, or no result yet."""
    return outcomes["testing"].get(test_id, (WARN, ""))[0] + moved("testing", [test_id])


def cesm(name):
    """How many CESM tests on this grid passed."""
    tests = [t for t in (runs or {}).get("tests", []) if t.get("grid") == name]
    return tally([OK if t["passed"] else FAIL for t in tests])


def performance_steps():
    """How many performance steps passed, every row and sweep."""
    found = outcomes["performance"]
    return tally([mark for mark, _ in found.values()]) + moved("performance", found)


rows = []
for name, grid in grids.items():
    rows.append(
        [
            name,
            checks.resolution(grid["grid"]),
            step(f"test_create[{name}]"),
            cesm(name) if grid.get("cesm_runs") else "",
        ]
    )
rows.append(["[Performance](performance.ipynb)", "", performance_steps(), ""])
text = checks.table(["Grid", "Res.", "Case", "CESM"], rows)
text += f"\n\n{OK} pass · {FAIL} fail · {NOT_RUN} skipped · {WARN} no result yet"
failed = [
    f"- `{t}`: {checks.plain(why)}"
    for found in outcomes.values()
    for t, (m, why) in found.items()
    if m == FAIL
]
if failed:
    text += "\n\n" + "\n".join(failed)
show_value = {"time": checks.fmt_s, "memory": checks.fmt_gb}
for suite, (past, changed) in changes.items():
    if not changed:
        continue
    found = [(t, *f) for t, fs in changed.items() for f in fs]
    rows = [
        [
            f"`{t}`",
            what,
            show_value[what](before),
            show_value[what](now),
            f"{SLOWER if now > before else FASTER} {now / before - 1:+.0%}",
        ]
        for t, what, before, now in sorted(
            found, key=lambda f: f[3] / f[2], reverse=True
        )
    ]
    text += (
        f"\n\n**{suite.capitalize()}:** {SLOWER} slower or bigger · {FASTER} faster or "
        + "smaller than the median of "
        + (f"the last {len(past)} runs" if len(past) > 1 else "the last run")
        + "\n\n"
        + checks.table(["Step", "", "Before", "Now", "Change"], rows)
    )
    was, now = past[0]["crocodash"], SUITES[suite]["crocodash"]
    if was and now and was["commit"] != now["commit"]:
        compare = f"https://github.com/CROCODILE-CESM/CrocoDash/compare/{was['commit']}...{now['commit']}"
        text += f"\n\nCrocoDash since the last run: [{was['commit']}…{now['commit']}]({compare})"
    elif was and now:
        text += f"\n\nSame CrocoDash commit as the last run ({now['commit']}), so not a code change."
checks.show(text)

(cesm)=
## CESM runs

MOM_interface's regional SMS tests, built with CrocoDash and run on Derecho, weekly.

In [ ]:
def phase_marks(test):
    """One mark per CIME phase, in order: passed, failed or not reached."""
    status = dict(test["phases"])
    marks = {"PASS": OK, "FAIL": FAIL}
    return "".join(marks.get(status.get(phase), NOT_RUN) for phase, _ in checks.PHASES)


def result(test):
    """'Pass', or the first phase that failed."""
    if test["passed"]:
        return f"{OK} Pass"
    status = dict(test["phases"])
    failed = [label for phase, label in checks.PHASES if status.get(phase) == "FAIL"]
    return f"{FAIL} {failed[0]}" if failed else f"{WARN} Incomplete"


if runs is None:
    checks.show("No results yet.")
else:
    # Failures first. Runs before the test matrix didn't record a grid.
    tests = sorted(
        runs["tests"], key=lambda t: (t["passed"], t.get("grid", ""), t["compset"])
    )
    rows = [
        [
            f"`{t['compset']}`",
            t.get("grid", "—"),
            f"`{t['test']}`",
            phase_marks(t),
            result(t),
        ]
        for t in tests
    ]
    order = " → ".join(label for _, label in checks.PHASES)
    checks.show(
        checks.table(["Compset", "Grid", "Test", "Phases", "Result"], rows)
        + f"\n\nPhases: {order}. CESM {checks.commit(runs.get('cesm'), 'CESM')}."
    )

(data-access)=
## Data access

Every way CrocoDash fetches data, called on Casper, weekly.

In [ ]:
if health is None:
    checks.show("No results yet.")
else:
    by_name = sorted(
        health["validate_checks"], key=lambda c: (c["product"], c["method"])
    )
    failing = [c for c in by_name if not c["ok"]]
    working = [c for c in by_name if c["ok"]]
    text = ""
    if failing:
        text += f"**{FAIL} Failing ({len(failing)})**\n\n" + checks.table(
            ["Product", "Access method", "Why", "Time"],
            [
                [
                    c["product"],
                    f"`{c['method']}`",
                    checks.plain(c.get("detail", "")),
                    checks.fmt_s(c["elapsed_s"]),
                ]
                for c in failing
            ],
        )
        text += "\n\n"
    text += f"**{OK} Working ({len(working)})**\n\n" + checks.table(
        ["Product", "Access method", "Time"],
        [
            [c["product"], f"`{c['method']}`", checks.fmt_s(c["elapsed_s"])]
            for c in working
        ],
    )
    checks.show(text)

(performance-summary)=
(mom6-scaling)=
## Performance

[Timings and fits](performance.ipynb) from the performance rows, run on the first Monday of
each month, and MOM6's speed against its task count.